In [1]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
import tqdm
import pickle
from torch_sparse import SparseTensor

In [3]:
p1 = np.load("68841_tweets_multiclasses_filtered_0722_part2.npy", allow_pickle=True)
p2 = np.load("68841_tweets_multiclasses_filtered_0722_part2.npy", allow_pickle=True)
g = np.concatenate((p1, p2), axis=0)
df = pd.DataFrame(data=g, columns=["event_id", "tweet_id", "text", "user_id", "created_at", "user_loc",
			"place_type", "place_full_name", "place_country_code", "hashtags", "user_mentions", "image_urls", "entities",
			"words", "filtered_words", "sampled_words"])

In [17]:
tweet_id2idx = {}
user_id2idx = {}
user = []
tweet = []
text_edges = []
text_nodes = [-1] * len(df) * 20
count = 0
node_labels = [-1] * len(df) * 20

# Use df instead of g for iteration
for _, row in tqdm.tqdm(df.iterrows()):
    # Convert tweet_id and user_id to string to ensure consistency
    tweet_id = str(row["tweet_id"])
    user_id = str(row["user_id"])

    if tweet_id not in tweet_id2idx:
        tweet_id2idx[tweet_id] = count
        tweet.append(count)
        count += 1
    else:
        tweet.append(tweet_id2idx[tweet_id])
    text_nodes[tweet_id2idx[tweet_id]] = (
        f"tweet"
    )
    node_labels[tweet_id2idx[tweet_id]] = str(row["event_id"])

    if user_id not in user_id2idx:
        user_id2idx[user_id] = count
        user.append(count)
        count += 1
    else:
        user.append(user_id2idx[user_id])
    text_nodes[user_id2idx[user_id]] = f"user"
    node_labels[user_id2idx[user_id]] = -1

    text_edges.append(row["text"])

    for mention in row["user_mentions"]:
        if mention not in user_id2idx:
            user_id2idx[mention] = count
            user.append(count)
            count += 1
        else:
            user.append(user_id2idx[mention])
        tweet.append(tweet_id2idx[tweet_id])
        text_nodes[user_id2idx[mention]] = f"mentioned user"
        node_labels[user_id2idx[mention]] = -1
        text_edges.append(row["text"])

67682it [00:04, 14848.15it/s]


In [18]:
text_nodes = text_nodes[:count]
node_labels = node_labels[:count]

In [19]:
edge_index = [user, tweet]
graph = Data(
			text_nodes=text_nodes,
			text_edges=text_edges,
			edge_index=torch.tensor(edge_index, dtype=torch.long),
			node_labels=node_labels
		)

In [20]:
with open('../processed/twitter.pkl', 'wb') as f:
    pickle.dump(graph, f)